# Virality Pipeline: Announcement vs Occurrence Analysis
Tests whether Camihawke audience bursts track the news-announcement date or the event-occurrence date.
Daily IG comment volume, SARIMAX counterfactual, paired Wilcoxon.
ASCII only. Outputs to Ali/outputs/virality/.

## Task 1 - Daily Series Build + Intervention Pairs

In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path

OUT = Path('outputs/virality')
OUT.mkdir(parents=True, exist_ok=True)
print('Output dir:', OUT.resolve())

Output dir: D:\Polythecninco di Milano\AFB_Lab\Ali\outputs\virality


In [2]:
# --- Load comment volume series (primary virality signal) ---
raw = pd.read_parquet('outputs/comments_ml.parquet')
print('comments_ml raw shape:', raw.shape, raw.dtypes[['platform','timestamp']].to_dict())

ig = raw[raw['platform'] == 'instagram'].copy()
print('IG rows:', len(ig))

ig['ts'] = pd.to_datetime(ig['timestamp'])
ig['date'] = ig['ts'].dt.date
print('date range:', ig['date'].min(), '->', ig['date'].max())

# Daily comment volume + distinct users + distinct posts
vol = (
    ig.groupby('date')
    .agg(
        n_comments=('comment_id', 'count'),
        n_active_users=('author_id', 'nunique'),
        n_active_posts=('media_id', 'nunique')
    )
    .reset_index()
)
vol['date'] = pd.to_datetime(vol['date'])
print('vol shape before reindex:', vol.shape)

comments_ml raw shape: (3661495, 20) {'platform': dtype('O'), 'timestamp': dtype('O')}
IG rows: 499752


date range: 2016-08-18 -> 2026-03-20


vol shape before reindex: (3480, 4)


In [3]:
# --- Load sentiment (secondary signal) ---
sent = pd.read_parquet('outputs/stage2_sentiment/sentiment_instagram.parquet')
print('sentiment shape:', sent.shape, sent.dtypes[['timestamp']].to_dict())

sent['ts'] = pd.to_datetime(sent['timestamp'])
sent['date'] = sent['ts'].dt.date

# Map toxicity string ordinal to numeric
tox_map = {'none': 0, 'low': 1, 'medium': 2, 'high': 3, 'very_high': 4}
sent['tox_num'] = sent['toxicity'].map(tox_map).fillna(0)

# Binary sentiment flags
sent['is_pos'] = (sent['sentiment_cat'] == 'positive').astype(int)
sent['is_neg'] = (sent['sentiment_cat'] == 'negative').astype(int)

sent_daily = (
    sent.groupby('date')
    .agg(
        mean_sentiment=('sentiment_score', 'mean'),
        pct_pos=('is_pos', 'mean'),
        pct_neg=('is_neg', 'mean'),
        mean_toxicity=('tox_num', 'mean')
    )
    .reset_index()
)
sent_daily['date'] = pd.to_datetime(sent_daily['date'])
print('sent_daily shape:', sent_daily.shape)

sentiment shape: (487604, 33) {'timestamp': dtype('O')}


sent_daily shape: (3478, 5)


In [4]:
# --- Full calendar reindex ---
date_min = pd.Timestamp('2016-08-18')
date_max = pd.Timestamp('2026-03-20')
all_dates = pd.DataFrame({'date': pd.date_range(date_min, date_max, freq='D')})

daily = all_dates.merge(vol, on='date', how='left')
daily = daily.merge(sent_daily, on='date', how='left')

# Fill volume with 0 (no activity), sentiment metrics with NaN
for c in ['n_comments', 'n_active_users', 'n_active_posts']:
    daily[c] = daily[c].fillna(0).astype(int)

daily['log1p_comments'] = np.log1p(daily['n_comments'])
daily = daily.set_index('date').sort_index()

print('daily series shape:', daily.shape)
print(daily.dtypes)
print('n_comments stats:', daily['n_comments'].describe().round(1))

daily series shape: (3502, 8)
n_comments          int64
n_active_users      int64
n_active_posts      int64
mean_sentiment    float64
pct_pos           float64
pct_neg           float64
mean_toxicity     float64
log1p_comments    float64
dtype: object
n_comments stats: count    3502.0
mean      142.7
std       319.4
min         0.0
25%        12.2
50%        42.0
75%       150.0
max      6633.0
Name: n_comments, dtype: float64


In [5]:
# Save daily series
daily.reset_index().to_parquet(OUT / 'daily_series.parquet', index=False)
print('Saved daily_series.parquet')

Saved daily_series.parquet


In [6]:
# --- Build intervention pairs ---
timeline = pd.read_csv('camihawke_combined_comprehensive_timeline.csv')
print('timeline shape:', timeline.shape)
print(timeline.dtypes)

timeline['Date'] = pd.to_datetime(timeline['Date'])
announcements = timeline[timeline['Entry_Type'] == 'First Public Announcement'].copy()
occurrences   = timeline[timeline['Entry_Type'] == 'Milestone / Occurrence'].copy()

print('\nAnnouncements:')
print(announcements[['Date','Event_Title','Category']].to_string())
print('\nOccurrences:')
print(occurrences[['Date','Event_Title','Category']].to_string())

timeline shape: (26, 6)
Date                  object
Event_Title           object
Entry_Type            object
Category              object
Details               object
Context_or_Metrics    object
dtype: object

Announcements:
         Date                                     Event_Title                     Category
2  2017-11-13                Pink Different (FoxLife TV Show)             Television Debut
3  2019-09-12                Nella MIA Cucina (Rai 2 TV Show)     Television Collaboration
6  2019-11-20                                 TEDxRimini 2019              Public Speaking
8  2021-03-09      Per tutto il resto dei miei sbagli (Novel)             Publishing Debut
10 2022-01-19  Tutte le volte che - Sanremo Edition (Podcast)  Audio / Spotify Partnership
12 2023-07-03        Il saggio di fine anno (Theatrical Tour)                Theater Debut
14 2024-06-14              Avanguardia Pura (Theatrical Tour)        Theater Collaboration
20 2025-04-10                               

In [7]:
# Manual pairing based on plan §2 guidance + data inspection
# Format: (milestone_label, announce_date, occurrence_date)
# announce_date=None => unpaired announcement; occurrence_date=None => unpaired occurrence

PAIRS = [
    # Announcement pairs
    ('Pink Different FoxLife',        '2017-11-13', None),         # No clear matching occurrence in window
    ('Nella MIA Cucina Rai2',         '2019-09-12', '2019-09-16'), # 4 days gap
    ('TEDxRimini 2019',               '2019-11-20', '2019-12-14'), # 24 days gap
    ('Novel Pre-Order',               '2021-03-09', '2021-04-20'), # 42 days gap
    ('Tutte le volte Podcast',        '2022-01-19', None),         # No matching occurrence
    ('Solo Tour Launch',              '2023-07-03', '2023-04-15'), # Announce AFTER occurrence (inverted)
    ('Avanguardia Pura Tour',         '2024-06-14', '2025-01-12'), # 212 days gap (tour premiere)
    ('Etna Comics 2025',              '2025-04-10', '2025-04-10'), # Same day (0 gap)
    ('The Traitors Italia S2',        '2025-11-07', '2026-06-12'), # 217 days gap
    # Occurrence-only (no announcement found)
    ('Digital Footprint Expansion',   None,         '2016-11-01'),
    ('Rai Radio2 Broadcast',          None,         '2017-09-15'),
    ('National TV Co-Hosting',        None,         '2019-09-16'),
    ('Macchianera Award',             None,         '2019-11-05'),
    ('Avanguardia Ticket Sales',      None,         '2024-06-19'),
    ('Avanguardia Winter Premiere',   None,         '2025-01-12'),
    ('Media Breakup Confirmation',    None,         '2025-02-24'),
    ('Autumn Tour Expansion',         None,         '2025-03-27'),
    ('Autumn Tour Re-ignition',       None,         '2025-10-19'),
    ('WMF Creators Fest',             None,         '2026-06-04'),
    ('Traitors Cast Unveiling',       None,         '2026-06-12'),
]

rows = []
for milestone, ann, occ in PAIRS:
    ann_dt = pd.Timestamp(ann) if ann else pd.NaT
    occ_dt = pd.Timestamp(occ) if occ else pd.NaT
    gap = int((occ_dt - ann_dt).days) if (pd.notna(ann_dt) and pd.notna(occ_dt)) else None
    paired = pd.notna(ann_dt) and pd.notna(occ_dt)
    # Flag inverted (announcement after occurrence)
    inverted = bool(gap is not None and gap < 0)
    rows.append({
        'milestone': milestone,
        'announce_date': ann_dt,
        'occurrence_date': occ_dt,
        'gap_days': gap,
        'paired': paired,
        'inverted': inverted,
    })

pairs_df = pd.DataFrame(rows)
print(pairs_df.to_string())

                      milestone announce_date occurrence_date  gap_days  paired  inverted
0        Pink Different FoxLife    2017-11-13             NaT       NaN   False     False
1         Nella MIA Cucina Rai2    2019-09-12      2019-09-16       4.0    True     False
2               TEDxRimini 2019    2019-11-20      2019-12-14      24.0    True     False
3               Novel Pre-Order    2021-03-09      2021-04-20      42.0    True     False
4        Tutte le volte Podcast    2022-01-19             NaT       NaN   False     False
5              Solo Tour Launch    2023-07-03      2023-04-15     -79.0    True      True
6         Avanguardia Pura Tour    2024-06-14      2025-01-12     212.0    True     False
7              Etna Comics 2025    2025-04-10      2025-04-10       0.0    True     False
8        The Traitors Italia S2    2025-11-07      2026-06-12     217.0    True     False
9   Digital Footprint Expansion           NaT      2016-11-01       NaN   False     False
10        

In [8]:
# Inseparability flags per post-window
for window in [7, 14, 28]:
    pairs_df[f'inseparable_{window}d'] = (
        pairs_df['paired'] &
        pairs_df['gap_days'].apply(lambda g: g is not None and abs(g) < window)
    )

# Coverage: does the series have pre-period data?
series_start = daily.index.min()
MIN_PRE = 90  # minimum days needed for SARIMAX baseline (90 days = ~13 weeks)

for dt_col in ['announce_date', 'occurrence_date']:
    pairs_df[f'{dt_col}_pre_days'] = pairs_df[dt_col].apply(
        lambda d: int((d - series_start).days) if pd.notna(d) else None
    )
    pairs_df[f'{dt_col}_feasible'] = pairs_df[f'{dt_col}_pre_days'].apply(
        lambda x: x is not None and x >= MIN_PRE
    )

print('Pairs with coverage flags:')
print(pairs_df[['milestone','gap_days','paired','inverted',
                'inseparable_7d','inseparable_14d','inseparable_28d',
                'announce_date_pre_days','announce_date_feasible',
                'occurrence_date_pre_days','occurrence_date_feasible'
               ]].to_string())

pairs_df.to_csv(OUT / 'intervention_pairs.csv', index=False)
print('\nSaved intervention_pairs.csv')

Pairs with coverage flags:
                      milestone  gap_days  paired  inverted  inseparable_7d  inseparable_14d  inseparable_28d  announce_date_pre_days  announce_date_feasible  occurrence_date_pre_days  occurrence_date_feasible
0        Pink Different FoxLife       NaN   False     False           False            False            False                   452.0                    True                       NaN                     False
1         Nella MIA Cucina Rai2       4.0    True     False            True             True             True                  1120.0                    True                    1124.0                      True
2               TEDxRimini 2019      24.0    True     False           False            False             True                  1189.0                    True                    1213.0                      True
3               Novel Pre-Order      42.0    True     False           False            False            False                  1664.0


Saved intervention_pairs.csv


In [9]:
# --- Overview figure: daily comment volume with announcement markers ---
fig, ax = plt.subplots(figsize=(18, 5))

ax.fill_between(daily.index, daily['n_comments'], alpha=0.35, color='steelblue', linewidth=0)
ax.plot(daily.index, daily['n_comments'], color='steelblue', linewidth=0.5, alpha=0.7)

# Mark announcement dates
ann_dates = pairs_df.dropna(subset=['announce_date'])[['milestone','announce_date']]
for _, row in ann_dates.iterrows():
    dt = row['announce_date']
    if dt in daily.index:
        y_max = daily.loc[dt, 'n_comments']
    else:
        y_max = 0
    ax.axvline(dt, color='crimson', linewidth=1.2, linestyle='--', alpha=0.8)
    ax.text(dt, daily['n_comments'].max()*0.82, row['milestone'][:18],
            rotation=90, fontsize=6, color='crimson', va='top')

ax.set_title('Daily IG Comment Volume with Announcement Dates', fontsize=12)
ax.set_xlabel('Date')
ax.set_ylabel('n_comments')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(OUT / 'fig_overview_daily_volume.png', dpi=150)
plt.close()
print('Saved fig_overview_daily_volume.png')

Saved fig_overview_daily_volume.png


## CHECKPOINT (end of Task 1)
Report: daily series shape, date range, coverage, intervention pairs table.

In [10]:
print('=== CHECKPOINT REPORT ===')
print(f'Daily series: {daily.shape[0]} rows x {daily.shape[1]} cols')
print(f'Date range: {daily.index.min().date()} -> {daily.index.max().date()}')
print(f'n_comments: median={daily["n_comments"].median():.0f}, max={daily["n_comments"].max()}')
print(f'Zero-volume days: {(daily["n_comments"]==0).sum()}')
print(f'\nTotal pairs rows: {len(pairs_df)}')
print(f'Fully paired (announce+occurrence): {pairs_df["paired"].sum()}')
print(f'Inverted pairs (announce after occ): {pairs_df["inverted"].sum()}')
print(f'Inseparable at 14-day window: {pairs_df["inseparable_14d"].sum()}')
print(f'\nAnnouncement dates with >=90d pre-period: {pairs_df["announce_date_feasible"].sum()}')
print(f'Occurrence dates with >=90d pre-period: {pairs_df["occurrence_date_feasible"].sum()}')
print('\nPer-pair gap summary:')
print(pairs_df[['milestone','gap_days','inseparable_14d','announce_date_feasible','occurrence_date_feasible']].to_string())

=== CHECKPOINT REPORT ===
Daily series: 3502 rows x 8 cols
Date range: 2016-08-18 -> 2026-03-20
n_comments: median=42, max=6633
Zero-volume days: 22

Total pairs rows: 20
Fully paired (announce+occurrence): 7
Inverted pairs (announce after occ): 1
Inseparable at 14-day window: 2

Announcement dates with >=90d pre-period: 9
Occurrence dates with >=90d pre-period: 17

Per-pair gap summary:
                      milestone  gap_days  inseparable_14d  announce_date_feasible  occurrence_date_feasible
0        Pink Different FoxLife       NaN            False                    True                     False
1         Nella MIA Cucina Rai2       4.0             True                    True                      True
2               TEDxRimini 2019      24.0            False                    True                      True
3               Novel Pre-Order      42.0            False                    True                      True
4        Tutte le volte Podcast       NaN            False      

## Task 2 - SARIMAX Counterfactual Per Intervention Date

In [11]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests
import itertools

# SARIMAX grid: small to keep runtime manageable
PDQS = list(itertools.product(
    [0,1,2],  # p
    [0,1],    # d
    [0,1,2],  # q
))
SEASONAL = [(1,0,1,7)]  # fixed seasonal order m=7 (weekly)

def best_sarimax(endog, seasonal_order=(1,0,1,7)):
    """Grid search over (p,d,q) with fixed seasonal, return best AIC model."""
    best_aic = np.inf
    best_model = None
    for p, d, q in PDQS:
        try:
            m = SARIMAX(
                endog,
                order=(p, d, q),
                seasonal_order=seasonal_order,
                enforce_stationarity=False,
                enforce_invertibility=False
            ).fit(disp=False, maxiter=200)
            if m.aic < best_aic:
                best_aic = m.aic
                best_model = m
        except Exception:
            pass
    return best_model


def run_counterfactual(
    intervention_date, label,
    lookback=365, post_window=14,
    series=None, col='log1p_comments'
):
    """
    Fit SARIMAX on pre-intervention window, forecast post_window days forward.
    Returns a dict of effect metrics.
    """
    if series is None:
        series = daily

    dt = pd.Timestamp(intervention_date)
    pre_start = dt - pd.Timedelta(days=lookback)
    pre_end   = dt - pd.Timedelta(days=1)
    post_end  = dt + pd.Timedelta(days=post_window - 1)

    pre  = series.loc[pre_start:pre_end, col].dropna()
    post = series.loc[dt:post_end, col]

    if len(pre) < 90:
        return {'milestone': label, 'intervention_date': dt,
                'post_window': post_window, 'lookback': lookback,
                'status': 'insufficient_pre',
                'cumulative_excess': np.nan, 'peak_excess': np.nan,
                'day_of_peak': np.nan, 'relative_lift': np.nan,
                'point_significance': np.nan, 'n_pre': len(pre)}

    model = best_sarimax(pre)
    if model is None:
        return {'milestone': label, 'intervention_date': dt,
                'post_window': post_window, 'lookback': lookback,
                'status': 'model_failed',
                'cumulative_excess': np.nan, 'peak_excess': np.nan,
                'day_of_peak': np.nan, 'relative_lift': np.nan,
                'point_significance': np.nan, 'n_pre': len(pre)}

    fc = model.get_forecast(steps=post_window)
    fc_mean = fc.predicted_mean.values
    ci = np.asarray(fc.conf_int())   # shape (post_window, 2)
    ci_lo, ci_hi = ci[:, 0], ci[:, 1]

    # Align observed post-period (may be shorter if near series end)
    obs_log = post.reindex(
        pd.date_range(dt, periods=post_window, freq='D')
    ).values
    valid = ~np.isnan(obs_log)

    excess_log = obs_log - fc_mean  # in log1p space

    # Back-transform cumulative excess: sum(expm1(obs)) - sum(expm1(fc))
    cum_excess_raw = np.sum(np.expm1(obs_log[valid])) - np.sum(np.expm1(fc_mean[valid]))
    rel_lift = (
        np.sum(np.expm1(obs_log[valid])) / np.sum(np.expm1(fc_mean[valid]))
        if np.sum(np.expm1(fc_mean[valid])) > 0 else np.nan
    )

    # Peak excess in log space
    peak_idx = int(np.argmax(excess_log[valid]))
    peak_excess_log = excess_log[valid][peak_idx]
    day_of_peak = peak_idx  # lag from intervention

    # Fraction of days where obs > 95% PI upper bound
    above_pi = np.sum(obs_log[valid] > ci_hi[valid]) / valid.sum() if valid.sum() > 0 else np.nan

    order_str = str(model.model.order) + 'x' + str(model.model.seasonal_order)
    return {
        'milestone': label,
        'intervention_date': dt,
        'post_window': post_window,
        'lookback': lookback,
        'status': 'ok',
        'n_pre': len(pre),
        'model_order': order_str,
        'aic': round(model.aic, 2),
        'cumulative_excess': round(cum_excess_raw, 1),
        'peak_excess_log': round(float(peak_excess_log), 4),
        'day_of_peak': day_of_peak,
        'relative_lift': round(float(rel_lift), 4) if not np.isnan(rel_lift) else np.nan,
        'point_significance': round(float(above_pi), 4),
        # Store arrays for figure
        '_fc_mean': fc_mean,
        '_ci_lo': ci_lo,
        '_ci_hi': ci_hi,
        '_obs_log': obs_log,
        '_pre': pre,
    }

print('Helper functions defined.')

Helper functions defined.


In [12]:
# Load post timestamps for new-post coincidence check
posts = pd.read_parquet('outputs/ig_multimodal_final.parquet',
                        columns=['media_id','timestamp','like_count'])
posts['date'] = pd.to_datetime(posts['timestamp']).dt.date
post_dates = set(str(d) for d in posts['date'].unique())

def new_post_coincidence(intervention_date, window=3):
    """Returns True if any new post landed within `window` days of intervention."""
    dt = pd.Timestamp(intervention_date)
    for lag in range(window + 1):
        if str((dt + pd.Timedelta(days=lag)).date()) in post_dates:
            return True
    return False

print('Post coincidence helper ready. Total posts:', len(posts))

Post coincidence helper ready. Total posts: 1493


In [13]:
# --- Run SARIMAX counterfactual for every feasible (milestone, date) pair ---
# We iterate over all unique feasible intervention dates
# Use two lookbacks (180, 365) and three post-windows (7, 14, 28)

intervention_candidates = []

for _, row in pairs_df.iterrows():
    for dt_col in ['announce_date', 'occurrence_date']:
        dt = row[dt_col]
        feasible = row[f'{dt_col}_feasible']
        if pd.isna(dt) or not feasible:
            continue
        # Skip if date is beyond series end
        if dt > daily.index.max():
            continue
        intervention_candidates.append({
            'milestone': row['milestone'],
            'intervention_date': dt,
            'intervention_type': 'announcement' if dt_col == 'announce_date' else 'occurrence',
            'gap_days': row['gap_days'],
        })

# Deduplicate (same date can appear as both announcement and occurrence)
seen = set()
unique_candidates = []
for c in intervention_candidates:
    key = (str(c['intervention_date']), c['intervention_type'])
    if key not in seen:
        seen.add(key)
        unique_candidates.append(c)

print(f'Total intervention candidates: {len(unique_candidates)}')
for c in unique_candidates:
    print(f"  {c['milestone'][:30]:30s} {c['intervention_type']:12s} {str(c['intervention_date'])[:10]}")

Total intervention candidates: 21
  Pink Different FoxLife         announcement 2017-11-13
  Nella MIA Cucina Rai2          announcement 2019-09-12
  Nella MIA Cucina Rai2          occurrence   2019-09-16
  TEDxRimini 2019                announcement 2019-11-20
  TEDxRimini 2019                occurrence   2019-12-14
  Novel Pre-Order                announcement 2021-03-09
  Novel Pre-Order                occurrence   2021-04-20
  Tutte le volte Podcast         announcement 2022-01-19
  Solo Tour Launch               announcement 2023-07-03
  Solo Tour Launch               occurrence   2023-04-15
  Avanguardia Pura Tour          announcement 2024-06-14
  Avanguardia Pura Tour          occurrence   2025-01-12
  Etna Comics 2025               announcement 2025-04-10
  Etna Comics 2025               occurrence   2025-04-10
  The Traitors Italia S2         announcement 2025-11-07
  Rai Radio2 Broadcast           occurrence   2017-09-15
  Macchianera Award              occurrence   2019-11-

In [14]:
# Main counterfactual loop
# Primary: 14-day window, 365-day lookback
# Also run 7d and 28d for robustness

all_results = []

for cand in unique_candidates:
    dt = cand['intervention_date']
    label = cand['milestone']
    itype = cand['intervention_type']

    for lookback in [180, 365]:
        for post_window in [7, 14, 28]:
            print(f'  Fitting: {label[:28]:28s} {itype:12s} lb={lookback} pw={post_window}d ... ', end='')
            res = run_counterfactual(dt, label, lookback=lookback, post_window=post_window)
            res['intervention_type'] = itype
            res['new_post_coincidence'] = new_post_coincidence(dt, window=3)
            # Inseparability flags from pairs_df
            match = pairs_df[pairs_df['milestone'] == label]
            res['inseparable_14d'] = bool(match['inseparable_14d'].values[0]) if len(match) else False
            all_results.append(res)
            print(f'  status={res["status"]}  excess={res.get("cumulative_excess", "N/A")}')

print(f'\nTotal result rows: {len(all_results)}')

  Fitting: Pink Different FoxLife       announcement lb=180 pw=7d ... 

D:\conda_envs\ma_env\lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
D:\conda_envs\ma_env\lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
D:\conda_envs\ma_env\lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
D:\conda_envs\ma_env\lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
D:\conda_envs\ma_env\lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, f

D:\conda_envs\ma_env\lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
D:\conda_envs\ma_env\lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
D:\conda_envs\ma_env\lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
D:\conda_envs\ma_env\lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
D:\conda_envs\ma_env\lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, f

D:\conda_envs\ma_env\lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
D:\conda_envs\ma_env\lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
D:\conda_envs\ma_env\lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
D:\conda_envs\ma_env\lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


D:\conda_envs\ma_env\lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
D:\conda_envs\ma_env\lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


...

D:\conda_envs\ma_env\lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
D:\conda_envs\ma_env\lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


D:\conda_envs\ma_env\lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
D:\conda_envs\ma_env\lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


D:\conda_envs\ma_env\lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
D:\conda_envs\ma_env\lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
D:\conda_envs\ma_env\lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
D:\conda_envs\ma_env\lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


D:\conda_envs\ma_env\lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
D:\conda_envs\ma_env\lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


  status=ok  excess=4227.4

Total result rows: 126


In [15]:
# Build exportable results dataframe (drop internal array columns)
drop_keys = ['_fc_mean', '_ci_lo', '_ci_hi', '_obs_log', '_pre']
export_rows = [{k: v for k, v in r.items() if k not in drop_keys} for r in all_results]
results_df = pd.DataFrame(export_rows)

# FDR correction on point_significance (fraction above PI) as a proxy p-value
# We treat point_significance as a "fraction significant" not a p-value;
# for formal FDR correction, use per-day binomial p (obs > PI_hi ~ Bernoulli(0.05 under H0))
# Simpler: flag rows where point_significance > 0.5 (majority of post days above PI)
results_df['majority_above_pi'] = results_df['point_significance'] > 0.5
results_df['strong_lift'] = results_df['relative_lift'] > 1.2  # 20% above counterfactual

print('results_df shape:', results_df.shape)
print(results_df.dtypes)
print(results_df[['milestone','intervention_type','post_window','lookback',
                   'status','cumulative_excess','relative_lift','point_significance',
                   'day_of_peak','majority_above_pi']].to_string())

results_df.to_csv(OUT / 'virality_results.csv', index=False)
print('\nSaved virality_results.csv')

results_df shape: (126, 18)
milestone                       object
intervention_date       datetime64[ns]
post_window                      int64
lookback                         int64
status                          object
n_pre                            int64
model_order                     object
aic                            float64
cumulative_excess              float64
peak_excess_log                float64
day_of_peak                      int64
relative_lift                  float64
point_significance             float64
intervention_type               object
new_post_coincidence              bool
inseparable_14d                   bool
majority_above_pi                 bool
strong_lift                       bool
dtype: object
                      milestone intervention_type  post_window  lookback status  cumulative_excess  relative_lift  point_significance  day_of_peak  majority_above_pi
0        Pink Different FoxLife      announcement            7       180     ok           

### Per-intervention figures (SARIMAX counterfactual)

In [16]:
# One figure per unique (milestone, intervention_type), primary window 14d, lookback 365d
PRIMARY_LB = 365
PRIMARY_PW = 14

primary_results = [
    r for r in all_results
    if r.get('lookback') == PRIMARY_LB
    and r.get('post_window') == PRIMARY_PW
    and r.get('status') == 'ok'
]

for res in primary_results:
    dt = res['intervention_date']
    label = res['milestone']
    itype = res['intervention_type']
    pre   = res['_pre']
    fc_mean = res['_fc_mean']
    ci_lo = res['_ci_lo']
    ci_hi = res['_ci_hi']
    obs_log = res['_obs_log']

    post_dates_idx = pd.date_range(dt, periods=PRIMARY_PW, freq='D')
    obs_raw   = np.expm1(np.where(np.isnan(obs_log), np.nan, obs_log))
    fc_raw    = np.expm1(fc_mean)
    ci_lo_raw = np.expm1(ci_lo)
    ci_hi_raw = np.expm1(ci_hi)

    # Show last 60 days of pre-period + post-window
    pre_plot = pre.iloc[-60:]

    fig, axes = plt.subplots(2, 1, figsize=(12, 7))

    # --- Top: raw comment counts ---
    ax = axes[0]
    ax.fill_between(pre_plot.index, np.expm1(pre_plot.values), alpha=0.4, color='steelblue')
    ax.plot(pre_plot.index, np.expm1(pre_plot.values), color='steelblue', lw=1, label='Observed (pre)')
    ax.fill_between(post_dates_idx, ci_lo_raw, ci_hi_raw, alpha=0.3, color='orange', label='95% PI')
    ax.plot(post_dates_idx, fc_raw, color='orange', lw=1.5, linestyle='--', label='SARIMAX forecast')
    ax.plot(post_dates_idx, obs_raw, color='crimson', lw=1.5, marker='o', markersize=3, label='Observed (post)')
    ax.axvline(dt, color='crimson', lw=1.5, label='Intervention')
    ax.set_title(f'{label}  |  {itype}  |  {str(dt)[:10]}', fontsize=10)
    ax.set_ylabel('n_comments')
    ax.legend(fontsize=7, loc='upper left')

    # --- Bottom: excess (obs - forecast) ---
    ax2 = axes[1]
    excess_raw = obs_raw - fc_raw
    ax2.bar(post_dates_idx, excess_raw, color=[
        'forestgreen' if e >= 0 else 'crimson' for e in excess_raw
    ], alpha=0.7)
    ax2.axhline(0, color='black', lw=0.8)
    ax2.set_ylabel('Excess comments (obs - forecast)')
    ax2.set_xlabel('Date')

    metrics_txt = (
        f'cum_excess={res["cumulative_excess"]:.0f}  '
        f'rel_lift={res["relative_lift"]:.3f}  '
        f'pct_above_PI={res["point_significance"]:.0%}  '
        f'day_of_peak={res["day_of_peak"]}'
    )
    ax2.set_title(metrics_txt, fontsize=8)

    plt.tight_layout()
    safe_label = label.replace(' ', '_').replace('/', '-')[:40]
    fname = OUT / f'fig_{itype[:3]}_{str(dt)[:10]}_{safe_label}.png'
    plt.savefig(fname, dpi=150)
    plt.close()
    print(f'Saved {fname.name}')

print('All primary figures saved.')

Saved fig_ann_2017-11-13_Pink_Different_FoxLife.png


Saved fig_ann_2019-09-12_Nella_MIA_Cucina_Rai2.png


Saved fig_occ_2019-09-16_Nella_MIA_Cucina_Rai2.png


Saved fig_ann_2019-11-20_TEDxRimini_2019.png


Saved fig_occ_2019-12-14_TEDxRimini_2019.png


...

Saved fig_occ_2019-11-05_Macchianera_Award.png


Saved fig_occ_2024-06-19_Avanguardia_Ticket_Sales.png


Saved fig_occ_2025-02-24_Media_Breakup_Confirmation.png


Saved fig_occ_2025-03-27_Autumn_Tour_Expansion.png


Saved fig_occ_2025-10-19_Autumn_Tour_Re-ignition.png
All primary figures saved.


## Task 3 - Announcement vs Occurrence: Paired Comparison + Wilcoxon

In [17]:
# For each PAIRED milestone, get the 14d/365lb excess for announcement and occurrence
primary_df = results_df[
    (results_df['post_window'] == 14) &
    (results_df['lookback'] == 365) &
    (results_df['status'] == 'ok')
].copy()

ann_df = primary_df[primary_df['intervention_type'] == 'announcement'][[
    'milestone', 'cumulative_excess', 'relative_lift', 'point_significance', 'day_of_peak'
]].rename(columns={
    'cumulative_excess': 'ann_excess',
    'relative_lift': 'ann_lift',
    'point_significance': 'ann_pt_sig',
    'day_of_peak': 'ann_peak_day'
})

occ_df = primary_df[primary_df['intervention_type'] == 'occurrence'][[
    'milestone', 'cumulative_excess', 'relative_lift', 'point_significance', 'day_of_peak'
]].rename(columns={
    'cumulative_excess': 'occ_excess',
    'relative_lift': 'occ_lift',
    'point_significance': 'occ_pt_sig',
    'day_of_peak': 'occ_peak_day'
})

compare = ann_df.merge(occ_df, on='milestone', how='inner')
compare['excess_diff'] = compare['ann_excess'] - compare['occ_excess']  # positive => ann > occ
compare['lift_diff']   = compare['ann_lift'] - compare['occ_lift']

# Add inseparable flag from pairs_df
compare = compare.merge(pairs_df[['milestone','inseparable_14d','gap_days','inverted']], on='milestone', how='left')
print('Paired comparison table:')
print(compare[['milestone','ann_excess','occ_excess','excess_diff','ann_lift','occ_lift','ann_peak_day','occ_peak_day','inseparable_14d','gap_days']].to_string())

Paired comparison table:
               milestone  ann_excess  occ_excess  excess_diff  ann_lift  occ_lift  ann_peak_day  occ_peak_day  inseparable_14d  gap_days
0  Nella MIA Cucina Rai2      3786.3      1641.2       2145.1    4.1664    2.3237             3             0             True       4.0
1        TEDxRimini 2019       247.6      1986.2      -1738.6    1.2186    3.0048            12             5            False      24.0
2        Novel Pre-Order      3819.0      1854.2       1964.8   11.1041    3.6236             6             0            False      42.0
3       Solo Tour Launch       -42.8      1229.6      -1272.4    0.9519    3.0310             1             0            False     -79.0
4  Avanguardia Pura Tour      1216.9       213.6       1003.3    3.0585    1.9227             7             8            False     212.0
5       Etna Comics 2025       429.4       429.4          0.0    2.0745    2.0745             4             4             True       0.0


In [18]:
from scipy.stats import wilcoxon

# Exclude inseparable pairs from the formal Wilcoxon (they cannot be distinguished)
separable = compare[~compare['inseparable_14d'] & ~compare['inverted']].copy()
print(f'Separable pairs for Wilcoxon: {len(separable)} (from {len(compare)} total paired)')
print(separable[['milestone','excess_diff']].to_string())

wilcoxon_result = None
wilcoxon_stat = np.nan
wilcoxon_p = np.nan

if len(separable) >= 3:  # need at least a few pairs for Wilcoxon
    diffs = separable['excess_diff'].values
    # Wilcoxon signed-rank: H0 = median difference is 0
    # H1 (one-sided) = announcement excess > occurrence excess (alternative='greater')
    try:
        stat, p_two = wilcoxon(diffs, alternative='two-sided', zero_method='wilcox')
        stat_gt, p_gt = wilcoxon(diffs, alternative='greater', zero_method='wilcox')
        wilcoxon_stat = stat
        wilcoxon_p = p_two
        print(f'\nWilcoxon signed-rank (H0: median diff=0):')
        print(f'  Two-sided:  stat={stat:.3f}, p={p_two:.4f}')
        print(f'  One-sided (ann > occ): stat={stat_gt:.3f}, p={p_gt:.4f}')
        print(f'  n_pairs={len(diffs)}, median_diff={np.median(diffs):.1f}')
    except Exception as e:
        print(f'Wilcoxon failed: {e}')
        wilcoxon_p = np.nan
else:
    print('Not enough separable pairs for Wilcoxon (need >= 3). Descriptive only.')

# Direction summary
ann_wins = (separable['excess_diff'] > 0).sum() if len(separable) > 0 else 0
occ_wins = (separable['excess_diff'] < 0).sum() if len(separable) > 0 else 0
print(f'\nDirection: announcement excess > occurrence: {ann_wins}/{len(separable)}')
print(f'Direction: occurrence excess > announcement: {occ_wins}/{len(separable)}')

Separable pairs for Wilcoxon: 3 (from 6 total paired)
               milestone  excess_diff
1        TEDxRimini 2019      -1738.6
2        Novel Pre-Order       1964.8
4  Avanguardia Pura Tour       1003.3

Wilcoxon signed-rank (H0: median diff=0):
  Two-sided:  stat=2.000, p=0.7500
  One-sided (ann > occ): stat=4.000, p=0.3750
  n_pairs=3, median_diff=1003.3

Direction: announcement excess > occurrence: 2/3
Direction: occurrence excess > announcement: 1/3


In [19]:
# Add aggregate result row
agg_row = pd.DataFrame([{
    'milestone': '_AGGREGATE_',
    'ann_excess': separable['ann_excess'].sum() if len(separable) else np.nan,
    'occ_excess': separable['occ_excess'].sum() if len(separable) else np.nan,
    'excess_diff': separable['excess_diff'].median() if len(separable) else np.nan,
    'ann_lift': np.nan, 'occ_lift': np.nan,
    'ann_peak_day': np.nan, 'occ_peak_day': np.nan,
    'inseparable_14d': False, 'gap_days': np.nan, 'inverted': False,
    'lift_diff': np.nan,
    'wilcoxon_stat': wilcoxon_stat,
    'wilcoxon_p_twosided': wilcoxon_p,
    'n_separable_pairs': len(separable),
    'ann_wins': ann_wins,
    'occ_wins': occ_wins,
}])

compare_out = pd.concat([compare, agg_row], ignore_index=True)
compare_out.to_csv(OUT / 'announce_vs_event.csv', index=False)
print('Saved announce_vs_event.csv')

Saved announce_vs_event.csv


## Task 4 - Robustness: Seasonal-Naive Baseline + ITS

In [20]:
# --- Seasonal-naive baseline (same-weekday median from the past 52 weeks) ---

def seasonal_naive_excess(intervention_date, label, itype, post_window=14, lookback=365):
    dt = pd.Timestamp(intervention_date)
    pre_start = dt - pd.Timedelta(days=lookback)
    pre_end   = dt - pd.Timedelta(days=1)

    pre_vol = daily.loc[pre_start:pre_end, 'n_comments']
    if len(pre_vol) < 90:
        return None

    pre_df = pre_vol.reset_index()
    pre_df.columns = ['date', 'n']
    pre_df['dow'] = pre_df['date'].dt.dayofweek
    dow_medians = pre_df.groupby('dow')['n'].median()

    rows = []
    for lag in range(post_window):
        d = dt + pd.Timedelta(days=lag)
        obs = daily.loc[d, 'n_comments'] if d in daily.index else np.nan
        fc  = dow_medians.get(d.dayofweek, np.nan)
        rows.append({'date': d, 'obs': obs, 'fc_naive': fc, 'excess_naive': obs - fc})

    naive_post = pd.DataFrame(rows)
    cum_excess = naive_post['excess_naive'].sum()
    rel_lift = (
        naive_post['obs'].sum() / naive_post['fc_naive'].sum()
        if naive_post['fc_naive'].sum() > 0 else np.nan
    )
    return {
        'milestone': label,
        'intervention_date': dt,
        'intervention_type': itype,
        'post_window': post_window,
        'naive_cumulative_excess': round(cum_excess, 1),
        'naive_relative_lift': round(rel_lift, 4),
    }

naive_rows = []
for cand in unique_candidates:
    dt = cand['intervention_date']
    if dt > daily.index.max():
        continue
    r = seasonal_naive_excess(dt, cand['milestone'], cand['intervention_type'])
    if r:
        naive_rows.append(r)

naive_df = pd.DataFrame(naive_rows)
print('Seasonal-naive robustness results:')
print(naive_df.to_string())

Seasonal-naive robustness results:
                     milestone intervention_date intervention_type  post_window  naive_cumulative_excess  naive_relative_lift
0       Pink Different FoxLife        2017-11-13      announcement           14                    359.0               2.8601
1        Nella MIA Cucina Rai2        2019-09-12      announcement           14                   3963.0               4.8891
2        Nella MIA Cucina Rai2        2019-09-16        occurrence           14                   1883.0               2.8868
3              TEDxRimini 2019        2019-11-20      announcement           14                    302.0               1.2801
4              TEDxRimini 2019        2019-12-14        occurrence           14                   1996.0               3.0347
5              Novel Pre-Order        2021-03-09      announcement           14                   2676.0               2.7594
6              Novel Pre-Order        2021-04-20        occurrence           14    

In [21]:
# --- ITS (Segmented OLS with HAC SEs) on daily log-volume around each intervention ---
import statsmodels.api as sm

def its_test(intervention_date, label, itype, pre_days=90, post_days=28):
    dt = pd.Timestamp(intervention_date)
    win_start = dt - pd.Timedelta(days=pre_days)
    win_end   = dt + pd.Timedelta(days=post_days)

    seg = daily.loc[win_start:win_end, 'log1p_comments'].reset_index()
    seg.columns = ['date', 'y']
    if len(seg) < 30:
        return None

    seg['t'] = (seg['date'] - dt).dt.days
    seg['D'] = (seg['t'] >= 0).astype(int)  # post indicator
    seg['tD'] = seg['t'] * seg['D']          # slope change

    X = sm.add_constant(seg[['t', 'D', 'tD']])
    try:
        ols = sm.OLS(seg['y'], X).fit(cov_type='HAC', cov_kwds={'maxlags': 7})
        level_jump = ols.params['D']
        level_p    = ols.pvalues['D']
        slope_chg  = ols.params['tD']
        slope_p    = ols.pvalues['tD']
        return {
            'milestone': label,
            'intervention_date': dt,
            'intervention_type': itype,
            'its_level_jump': round(level_jump, 5),
            'its_level_p': round(level_p, 5),
            'its_slope_chg': round(slope_chg, 5),
            'its_slope_p': round(slope_p, 5),
            'its_nobs': int(ols.nobs),
        }
    except Exception as e:
        return None

its_rows = []
for cand in unique_candidates:
    dt = cand['intervention_date']
    if dt > daily.index.max():
        continue
    r = its_test(dt, cand['milestone'], cand['intervention_type'])
    if r:
        its_rows.append(r)

its_df = pd.DataFrame(its_rows)
print('ITS results:')
print(its_df.to_string())

# FDR correction on ITS level-jump p-values
if len(its_df) > 0:
    reject, q_vals, _, _ = multipletests(its_df['its_level_p'].fillna(1), method='fdr_bh')
    its_df['its_level_q'] = q_vals
    its_df['its_level_sig'] = reject
    print('\nAfter BH-FDR correction on level-jump p-values:')
    print(its_df[['milestone','intervention_type','its_level_jump','its_level_p','its_level_q','its_level_sig']].to_string())

ITS results:
                     milestone intervention_date intervention_type  its_level_jump  its_level_p  its_slope_chg  its_slope_p  its_nobs
0       Pink Different FoxLife        2017-11-13      announcement         0.63814      0.13671       -0.01019      0.70248       119
1        Nella MIA Cucina Rai2        2019-09-12      announcement         1.15620      0.00486       -0.10611      0.00000       119
2        Nella MIA Cucina Rai2        2019-09-16        occurrence         0.16027      0.70432       -0.04736      0.12559       119
3              TEDxRimini 2019        2019-11-20      announcement        -1.83768      0.00407        0.10529      0.00006       119
4              TEDxRimini 2019        2019-12-14        occurrence         1.27478      0.02442       -0.04335      0.04271       119
5              Novel Pre-Order        2021-03-09      announcement         1.24604      0.01865       -0.02529      0.35283       119
6              Novel Pre-Order        2021-04-20 

In [22]:
# Merge naive + ITS into results and save
robust_df = naive_df.merge(its_df, on=['milestone','intervention_date','intervention_type'], how='outer')
robust_df.to_csv(OUT / 'robustness_results.csv', index=False)
print('Saved robustness_results.csv')
print(robust_df[['milestone','intervention_type','naive_relative_lift',
                  'its_level_jump','its_level_p','its_level_q']].to_string())

Saved robustness_results.csv
                     milestone intervention_type  naive_relative_lift  its_level_jump  its_level_p  its_level_q
0        Autumn Tour Expansion        occurrence               2.1342         0.75879      0.06460     0.123327
1      Autumn Tour Re-ignition        occurrence               8.7975         2.81456      0.00000     0.000000
2        Avanguardia Pura Tour      announcement               4.0089         0.39200      0.32765     0.430041
3        Avanguardia Pura Tour        occurrence               1.0749         0.71842      0.10812     0.189210
4     Avanguardia Ticket Sales        occurrence               3.3604        -0.27315      0.50519     0.589388
5             Etna Comics 2025      announcement               2.0319        -0.96031      0.01553     0.043517
6             Etna Comics 2025        occurrence               2.0319        -0.96031      0.01553     0.043517
7            Macchianera Award        occurrence               3.1059      

## Task 5 - Final Summary Markdown

In [23]:
# Build final summary
from datetime import date

# Pull key stats
n_candidates = len(unique_candidates)
n_paired_sep = len(separable)
n_inseparable = compare['inseparable_14d'].sum() if len(compare) > 0 else 0

primary_ok = results_df[
    (results_df['post_window'] == 14) &
    (results_df['lookback'] == 365) &
    (results_df['status'] == 'ok')
]

# Milestones where majority of post-window days were above PI (notable)
notable = primary_ok[primary_ok['majority_above_pi']]

lines = [
    '# Virality Analysis: Announcement vs Occurrence',
    f'Generated: {date.today()}',
    '',
    '## Hypothesis',
    'The audience burst tracks the NEWS announcement date, not the event-occurrence date.',
    'Tested via SARIMAX counterfactual daily comment volume; paired Wilcoxon signed-rank.',
    '',
    '## Data',
    '- Daily IG comment volume: 2016-08-18 to 2026-03-20 (3,480 distinct days, median=42, max=6,633)',
    '- Timeline: 9 First Public Announcements + 17 Milestones/Occurrences from combined CSV',
    '',
    '## Intervention Pairs Summary',
]

for _, row in pairs_df[pairs_df['paired']].iterrows():
    ins = 'INSEPARABLE (14d)' if row['inseparable_14d'] else 'separable'
    inv = ' [INVERTED: announce after occurrence]' if row['inverted'] else ''
    lines.append(
        f'- {row["milestone"]}: announce={str(row["announce_date"])[:10]},'
        f' occur={str(row["occurrence_date"])[:10]}, gap={row["gap_days"]}d ({ins}){inv}'
    )

lines += [
    '',
    '## Test 1 - SARIMAX Counterfactual (14-day window, 365-day lookback)',
    f'Total intervention dates tested: {n_candidates}',
    f'Dates with majority of post-window days above 95% PI: {len(notable)}',
]

if len(notable) > 0:
    for _, row in notable.iterrows():
        lines.append(
            f'  - {row["milestone"]} ({row["intervention_type"]}):'
            f' cum_excess={row["cumulative_excess"]:.0f},'
            f' lift={row["relative_lift"]:.3f},'
            f' pct_above_PI={row["point_significance"]:.0%},'
            f' day_of_peak={row["day_of_peak"]}'
        )
else:
    lines.append('  No intervention date had majority post-window days above 95% PI.')

lines += [
    '',
    '## Test 2 - Announcement vs Occurrence (Paired Wilcoxon)',
    f'Separable paired milestones: {n_paired_sep}',
    f'Inseparable (gap < 14d): {n_inseparable}',
]

if not np.isnan(wilcoxon_p):
    lines.append(f'Wilcoxon signed-rank (two-sided): stat={wilcoxon_stat:.3f}, p={wilcoxon_p:.4f}')
    lines.append(f'Announcement excess > occurrence: {ann_wins}/{n_paired_sep} separable pairs')
    if wilcoxon_p < 0.05:
        lines.append('Result: SIGNIFICANT difference in direction favoring announcements.')
    else:
        lines.append('Result: No significant systematic advantage for announcement dates (p >= 0.05).')
else:
    lines.append('Wilcoxon: not enough separable pairs (n < 3) for formal test. Descriptive only.')
    lines.append(f'Announcement excess > occurrence: {ann_wins}/{n_paired_sep} separable pairs')

lines += [
    '',
    '## Test 3 - Robustness (Seasonal-Naive + ITS)',
]

if len(its_df) > 0 and 'its_level_sig' in its_df.columns:
    sig_its = its_df[its_df['its_level_sig']]
    lines.append(f'ITS level-jump BH-FDR significant interventions: {len(sig_its)}')
    if len(sig_its) > 0:
        for _, row in sig_its.iterrows():
            lines.append(
                f'  - {row["milestone"]} ({row["intervention_type"]}):'
                f' jump={row["its_level_jump"]:.4f}, q={row["its_level_q"]:.4f}'
            )
    else:
        lines.append('  No intervention passed ITS level-jump after BH correction.')
else:
    lines.append('ITS could not be computed.')

lines += [
    '',
    '## Caveats',
    '1. Single subject (Camihawke), observational, no control creator. Counterfactual rests on pre-trend.',
    '2. Comment data only covers posts scraped — activity on unscraped posts is uncounted.',
    '3. Daily comment counts may reflect old posts re-activated, not new-post virality; cross-check n_active_posts.',
    '4. Inseparable pairs (gap < 14d) cannot distinguish announcement from occurrence burst.',
    '5. Heavy-tailed volume (max=6,633) means SARIMAX PIs can be wide; model on log1p to reduce heteroscedasticity.',
    '6. With only 9 pairs (fewer separable), Wilcoxon power is limited; interpret descriptively.',
    '',
    '## Outputs',
    '- outputs/virality/daily_series.parquet',
    '- outputs/virality/intervention_pairs.csv',
    '- outputs/virality/virality_results.csv',
    '- outputs/virality/announce_vs_event.csv',
    '- outputs/virality/robustness_results.csv',
    '- outputs/virality/fig_overview_daily_volume.png',
    '- outputs/virality/fig_ann_*.png (per-announcement SARIMAX figure)',
    '- outputs/virality/fig_occ_*.png (per-occurrence SARIMAX figure)',
]

summary_text = '\n'.join(lines)
with open(OUT / 'virality_summary.md', 'w', encoding='ascii', errors='replace') as f:
    f.write(summary_text)
print('Saved virality_summary.md')
print(summary_text)

Saved virality_summary.md
# Virality Analysis: Announcement vs Occurrence
Generated: 2026-06-14

## Hypothesis
The audience burst tracks the NEWS announcement date, not the event-occurrence date.
Tested via SARIMAX counterfactual daily comment volume; paired Wilcoxon signed-rank.

## Data
- Daily IG comment volume: 2016-08-18 to 2026-03-20 (3,480 distinct days, median=42, max=6,633)
- Timeline: 9 First Public Announcements + 17 Milestones/Occurrences from combined CSV

## Intervention Pairs Summary
- Nella MIA Cucina Rai2: announce=2019-09-12, occur=2019-09-16, gap=4.0d (INSEPARABLE (14d))
- TEDxRimini 2019: announce=2019-11-20, occur=2019-12-14, gap=24.0d (separable)
- Novel Pre-Order: announce=2021-03-09, occur=2021-04-20, gap=42.0d (separable)
- Solo Tour Launch: announce=2023-07-03, occur=2023-04-15, gap=-79.0d (separable) [INVERTED: announce after occurrence]
- Avanguardia Pura Tour: announce=2024-06-14, occur=2025-01-12, gap=212.0d (separable)
- Etna Comics 2025: announce=2025-04

In [24]:
import os
files = sorted(os.listdir(str(OUT)))
print(f'Files in {OUT}:')
for f in files:
    path = OUT / f
    sz = os.path.getsize(path)
    print(f'  {f:60s}  {sz:>10,} bytes')

Files in outputs\virality:
  announce_vs_event.csv                                                994 bytes
  daily_series.parquet                                             113,475 bytes
  fig_ann_2017-11-13_Pink_Different_FoxLife.png                    117,243 bytes
  fig_ann_2019-09-12_Nella_MIA_Cucina_Rai2.png                     131,211 bytes
  fig_ann_2019-11-20_TEDxRimini_2019.png                           124,597 bytes
  fig_ann_2021-03-09_Novel_Pre-Order.png                           124,441 bytes
  fig_ann_2022-01-19_Tutte_le_volte_Podcast.png                    115,106 bytes
  fig_ann_2023-07-03_Solo_Tour_Launch.png                          124,051 bytes
  fig_ann_2024-06-14_Avanguardia_Pura_Tour.png                     116,358 bytes
  fig_ann_2025-04-10_Etna_Comics_2025.png                          118,293 bytes
  fig_ann_2025-11-07_The_Traitors_Italia_S2.png                    120,713 bytes
  fig_occ_2017-09-15_Rai_Radio2_Broadcast.png                      134,235 bytes
 